# Compartment - gauss-only pipeline (Kaggle) - train80

Chạy **train80** (compound-level 80/20 split) với gói mã nguồn tinh gọn src/: chỉ gồm Gauss head (N(mu, sigma^2)), LoRA, 3 exits riêng cho Mod / Head / PV.

### Pipeline 2 Giai đoạn (Task Adaptation Prefix MLM & Downstream Dual Readout):
1. **Giai đoạn 1 (Task Adaptation - Prefix MLM):**
   - Đóng băng các tầng dưới, gắn LoRA cho tầng 18-22 và MLM Head.
   - Huấn luyện Prefix Prompt [CLS] <marker> [MASK] <marker> Context [SEP] với cơ chế **Stochastic 80/10/10 Denoising**.
   - Biến vị trí Prefix thành **Context Sink** (Learned Query Vector) chuyên biệt cho từng task.
   - Tự động gập trọng số LoRA (merge_lora) và lưu thành checkpoint Hugging Face chuẩn tại checkpoints/prefix_mlm_adapted.
2. **Giai đoạn 2 (Downstream Fine-Tuning - Dual Readout):**
   - Nạp checkpoint mmBERT đã thích ứng từ Giai đoạn 1.
   - Đưa từ thật vào Prefix, áp dụng **Dual Readout (DualGate)** kết hợp cả vector nén tại Prefix và span thực tế trong Context.
   - Warm-up MLP Head 3 epochs và mở LoRA 15 epochs để đạt hệ số tương quan Spearman rho cao nhất.

### Điều khiển nhanh:
- Chạy trọn vẹn cả 2 giai đoạn: giữ nguyên mặc định RUN_STAGE1_MLM = True và RUN_STAGE2_DOWNSTREAM = True.
- Chỉ chạy Giai đoạn 2 trực tiếp (không qua MLM adapt): đặt RUN_STAGE1_MLM = False (sẽ tự động dùng config/best_dual_gauss.json).


In [ ]:
import os, subprocess, sys
from pathlib import Path

def _importable(name: str) -> bool:
    try:
        __import__(name)
        return True
    except Exception:
        return False

# ---- repo source -----------------------------------------------------------
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/AmnO-O/MoTune.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')    # set for private repos

dest = Path('/kaggle/working/Compartment')
url = REPO_URL
if GITHUB_TOKEN:
    url = url.replace('https://', f'https://{GITHUB_TOKEN}@')
dest.parent.mkdir(parents=True, exist_ok=True)
if (dest / 'src' / 'run.py').is_file():
    print('refreshing existing clone at', dest)
    subprocess.run(['git', '-C', str(dest), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(dest), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    print('cloning', REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, url, str(dest)], check=True)
REPO = dest
os.chdir(REPO)
print('repo:', REPO)
print('head:', subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

# ---- pipeline stages & config selection ------------------------------------
def _env_or_secret(name: str, default: str = '') -> str:
    v = os.environ.get(name, '')
    if v:
        return v
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return default

# Stage 1: Task-Adaptive Prefix MLM pre-training (Context Sink)
RUN_STAGE1_MLM = _env_or_secret('RUN_STAGE1_MLM', 'true').lower() in ('1', 'true', 'yes')
CONFIG_STAGE1 = _env_or_secret('CONFIG_STAGE1', 'config/mlm_adapt.json')

# Stage 2: Downstream Dual Readout Fine-tuning
RUN_STAGE2_DOWNSTREAM = _env_or_secret('RUN_STAGE2_DOWNSTREAM', 'true').lower() in ('1', 'true', 'yes')
CONFIG_FILE = _env_or_secret('CONFIG_FILE', 'config/best_dual_adapted.json')

# ---- knobs ----------------------------------------------------------------
MODE = os.environ.get('MODE', 'all')      # all | train80
if MODE not in ('all', 'train80'):
    MODE = 'train80'
TRAIN_EXTRA = os.environ.get('TRAIN_EXTRA', '')   # extra --set flags, comma-separated

# ---- tunable config overrides ('' = keep config JSON default) -------------
# Edit values here ONLY if you want to override the JSON config file.
OVERRIDES = {
    'freeze_epochs': '',
    'lora_epochs': '',
    'lora_rank': '',
    'lora_alpha': '',
    'lora_from_layer': '',
    'batch_size': '',
    'accum_steps': '',
    'head_lr': '',
    'encoder_lr': '',
    'ccc_weight': '',
    'lambda_rank': '',
    'bin_sigma': '',
    'num_workers': '0',                      # 0 prevents Kaggle multiprocessing deadlock
    'target_prefix': '',
    'prefix_readout': '',
}


def cfg_sets(*extra):
    out = []
    for k, v in OVERRIDES.items():
        if v not in (None, ''):
            out += ['--set', f'{k}={v}']
    for e in extra:
        out += ['--set', e]
    return out

# ---- data mix: lineages tham gia train80 -----------------------------------
TRAIN_MIX = os.environ.get('TRAIN_MIX', 'full')
EXTRA_SETS: list = []
if TRAIN_MIX == 'nn' and not OVERRIDES.get('en_pv_train'):
    EXTRA_SETS += ['en_pv_train=', 'de_pv_train=']
if TRAIN_MIX in ('nn', 'default') and not OVERRIDES.get('de_pv_train') and 'de_pv_train=' not in EXTRA_SETS:
    EXTRA_SETS.append('de_pv_train=')

# ---- memory ---------------------------------------------------------------
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ---- HuggingFace push (optional) -----------------------------------------
HF_REPO_ID = os.environ.get('HF_REPO_ID', 'AmnO-O/compartment-weights')
HF_PRIVATE = os.environ.get('HF_PRIVATE', 'true').lower() not in ('0','false','no')

def _hf_token() -> str:
    tok = os.environ.get('HF_TOKEN', '')
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        return ''

# ---- ensure imports exist -------------------------------------------------
for m in ('torch', 'transformers', 'pandas', 'numpy', 'sklearn', 'scipy', 'yaml'):
    if not _importable(m):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', m], check=False)

def run(args):
    print('\n>>>', ' '.join(args))
    subprocess.run(args, cwd=REPO, check=True)



In [ ]:
# --- Static anchors: build the needed external vec once per session -----------
# Builder is chosen by static_ext_path's FILENAME:
#   cc_en_de_300.vec  -> build_static_vec.py     (fastText EN+DE, 300-d)
# Tat static cho A/B baseline: TRAIN_EXTRA='static_span=false,static_ext_path='
_flag_parts = [p for p in (os.environ.get('TRAIN_EXTRA', '') + ' '
               + ' '.join(f'{k}={v}' for k, v in OVERRIDES.items() if v))
              .replace(',', ' ').split()]
_cfg_static_path = ''
if CONFIG_FILE and Path(CONFIG_FILE).is_file():
    try:
        import json as _json
        _cfg_static_path = _json.load(open(CONFIG_FILE, encoding='utf-8')).get('static_ext_path', '') or ''
    except Exception:
        _cfg_static_path = ''
_pos = [p.split('=', 1)[1] for p in _flag_parts
        if p.startswith('static_ext_path=') and len(p) > len('static_ext_path=')]
STATIC_VEC_PATH = Path(_pos[0]) if _pos else (Path(_cfg_static_path) if _cfg_static_path
                                              else Path('/kaggle/working/cc_en_de_300.vec'))
_static_cfg = bool(CONFIG_FILE and 'static' in CONFIG_FILE)
_static_on = bool(_pos)
_static_off = ('static_span=false' in _flag_parts) or ('static_ext_path=' in _flag_parts)
_builder_map = {
    'cc_en_de_300.vec': REPO / 'scripts' / 'build_static_vec.py',
}

if MODE in ('all', 'train80') and (_static_cfg or _static_on) and not _static_off:
    script = _builder_map.get(STATIC_VEC_PATH.name)
    if not STATIC_VEC_PATH.is_file():
        if script is None:
            raise SystemExit('no builder registered for ' + STATIC_VEC_PATH.name
                             + ' in cell 2 _builder_map.')
        if not script.is_file():
            raise SystemExit(script.name + ' not in repo - commit & push it '
                             '(this notebook refreshes the clone at cell 1).')
        print('\n>>> building', STATIC_VEC_PATH.name, '...')
        subprocess.run([sys.executable, str(script), '--out', str(STATIC_VEC_PATH)],
                       cwd=REPO, check=True)
    size = f'{STATIC_VEC_PATH.stat().st_size / 1e6:.2f} MB' if STATIC_VEC_PATH.is_file() else 'MISSING'
    print('static vec:', STATIC_VEC_PATH, size)
elif MODE in ('all', 'train80'):
    print('static vec: skipped (khong dung static config / da tat static).')


In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# =========================================================================
# STAGE 1: Task-Adaptive Prefix MLM Pre-training (Context Sink)
# =========================================================================
if RUN_STAGE1_MLM and MODE in ('all', 'train80'):
    print('\n' + '=' * 70)
    print('>>> STARTING STAGE 1: Task-Adaptive Prefix MLM Pre-training')
    print('=' * 70)
    stage1_args = [
        sys.executable, '-m', 'src.run', '--device', 'cuda', '--mlm-adapt',
        '--config', CONFIG_STAGE1,
        '--set', 'num_workers=0',
    ]
    run(stage1_args)
    print('>>> STAGE 1 COMPLETED: Adapted backbone exported to checkpoints/prefix_mlm_adapted')

# =========================================================================
# STAGE 2: Downstream Dual Readout Fine-Tuning
# =========================================================================
if RUN_STAGE2_DOWNSTREAM and MODE in ('all', 'train80'):
    print('\n' + '=' * 70)
    print('>>> STARTING STAGE 2: Downstream Dual Readout Fine-Tuning')
    print('=' * 70)

    chosen_cfg = CONFIG_FILE
    adapted_dir = Path('checkpoints/prefix_mlm_adapted')
    # If the adapted checkpoint does not exist, fall back to best_dual_gauss.json
    if 'adapted' in chosen_cfg and not adapted_dir.is_dir():
        print('Warning: %s not found; falling back to config/best_dual_gauss.json' % adapted_dir)
        chosen_cfg = 'config/best_dual_gauss.json'
    elif not Path(chosen_cfg).is_file():
        chosen_cfg = 'config/best_dual_gauss.json'

    stage2_args = [
        sys.executable, '-m', 'src.run', '--device', 'cuda',
        '--config', chosen_cfg,
        '--set', 'num_workers=0',
    ]

    stage2_args += list(cfg_sets(*EXTRA_SETS))

    if TRAIN_EXTRA:
        stage2_args += [x.strip() for x in TRAIN_EXTRA.split(',') if x.strip()]

    # Guarantee static vec exists if configured
    _need = ''
    for _a in stage2_args:
        if _a.startswith('static_ext_path='):
            _v = _a.split('=', 1)[1]
            if _v:
                _need = _v
    if _need:
        _svp = Path(_need)
        if not _svp.is_file():
            _bsp = REPO / 'scripts' / 'build_static_vec.py'
            if _bsp.is_file():
                print('\n>>> building', _svp.name, '...')
                subprocess.run([sys.executable, str(_bsp), '--out', str(_svp)], cwd=REPO, check=True)

    print('>>> Running Stage 2 with config:', chosen_cfg)
    run(stage2_args)



In [ ]:
# --- Push trained weights & artifacts to HuggingFace ----------------------
if HF_REPO_ID and MODE in ('all', 'train80'):
    if not _hf_token():
        print('SKIP push: no HF_TOKEN found (set Kaggle Secret named HF_TOKEN).')
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=_hf_token())
        api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True, repo_type='model')
        work = Path('/kaggle/working')

        # 1. Upload model checkpoints
        models_dir = work / 'models'
        if models_dir.is_dir():
            api.upload_folder(
                folder_path=str(models_dir),
                repo_id=HF_REPO_ID,
                path_in_repo='models',
                allow_patterns='*.pt'
            )
            pt_files = list(models_dir.glob('*.pt'))
            print(f'pushed {len(pt_files)} model file(s)')

        # 2. Upload adapted MLM backbone if created in Stage 1
        adapted_dir = Path('checkpoints/prefix_mlm_adapted')
        if not adapted_dir.is_dir():
            adapted_dir = work / 'checkpoints' / 'prefix_mlm_adapted'
        if adapted_dir.is_dir():
            api.upload_folder(
                folder_path=str(adapted_dir),
                repo_id=HF_REPO_ID,
                path_in_repo='prefix_mlm_adapted',
            )
            print('pushed prefix_mlm_adapted backbone checkpoint')

        # 3. Upload key artifacts
        for name in ('config.json', 'metrics.json', 'history.json'):
            src = work / name
            if src.is_file():
                api.upload_file(path_or_fileobj=str(src), path_in_repo=name, repo_id=HF_REPO_ID)
                print(f'pushed {name}')
        print(f'HF repo: https://huggingface.co/{HF_REPO_ID}')
else:
    print('push skipped (HF_REPO_ID empty or MODE=%s)' % MODE)



In [ ]:
import json

work = Path('/kaggle/working')
p = work / 'metrics.json'
if p.is_file():
    print('--- metrics.json (val rho on holdout 20%) ---')
    print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2))
else:
    print('no metrics.json yet')


In [ ]:
# --- TRIAL after training: reload best.pt, score the REAL per-lineage trial files ----
# TRAI files la holdout rieng tung lineage (en-nn-trial / en-pv-trial / de-nn-trial /
# de-pv-trial). KHONG re-split train de lam trial - each lineage has its own file.
# Ghi submission theo DUNG format: zip gom cac file [language]-[task]-pred.tsv,
# KHONG header, moi dong: ContextID <tab> score(s) (NN: mod + head; PV: 1 score = mean
# cua 2 head). Moi lineage mot file pred rieng. Tu dong BO QUA neu nochua best.pt.
import sys, json, logging, torch, zipfile
from pathlib import Path

work = Path('/kaggle/working')
ckpt = work / 'models' / 'best.pt'
if not ckpt.is_file():
    for cand in (REPO / 'models' / 'best.pt', REPO / 'output' / 'models' / 'best.pt', Path('output/models/best.pt'), Path('models/best.pt')):
        if cand.is_file():
            ckpt = cand
            work = cand.parent.parent
            break
if not ckpt.is_file():
    print('SKIP trial: chua co models/best.pt (train chua xong).')
else:
    sys.path.insert(0, str(REPO))
    from src.config import Config
    from src.pipeline import _tokenizer
    from src.data import load_trial, CompDataset, collate_comp
    from src.model import apply_lora, build_model
    from src.train import evaluate
    from torch.utils.data import DataLoader
    from scipy.stats import spearmanr

    lg = logging.getLogger('trial')
    cfg_p = work / 'config.json'
    cfg = (Config() if not cfg_p.is_file()
           else Config(**json.loads(cfg_p.read_text(encoding='utf-8'))))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok = _tokenizer(cfg, lg)

    # best.pt duoc luu KHI LoRA adapters dang active (trainer._apply_lora wrap
    # top layers TRUOC khi train) -> phai wrap lai nhu the TRUOC load_state_dict,
    # nguoc lai moi lora key se mismatch (RuntimeError missing/unexpected).
    model = build_model(cfg, device)
    apply_lora(model, rank=cfg.lora_rank, alpha=cfg.lora_alpha,
               dropout=cfg.lora_dropout, targets=cfg.lora_targets,
               from_layer=cfg.lora_from_layer)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    lineage = load_trial(cfg)
    sub = work / 'submission'
    sub.mkdir(parents=True, exist_ok=True)
    for key, rows in lineage.items():
        # --- external static anchors: same wiring as trainer ------------------
        static_vec = None
        _sep = (getattr(cfg, 'static_ext_path', None) or '').strip()
        if _sep:
            if not Path(_sep).is_file():
                raise FileNotFoundError('static_ext_path not found: ' + _sep)
            from src.static_vec import StaticVec
            words = set()
            for r in rows:
                for k in ('mod', 'head', 'compound'):
                    w = r.get(k)
                    if w:
                        words.add(w)
            static_vec = StaticVec(_sep, cfg.static_ext_dim, words)
        ds = CompDataset(rows, tok, max_len=cfg.max_context_length,
                         static_vec=static_vec, target_prefix=cfg.target_prefix,
                         span_markers=getattr(cfg, 'span_markers', False))
        loader = DataLoader(ds, batch_size=64, shuffle=False, collate_fn=collate_comp)
        # return_all=True -> (mod, head, mod_y, head_y, mask), row-aligned voi rows.
        mp, hp, pp, my, hy, mask = evaluate(model, loader, device, return_all=True, return_pv=True)
        assert len(mp) == len(rows), (key, len(mp), len(rows))
        pv = bool(rows[0]['is_pv'])
        score = pp if pv else mp   # dedicated overall PV head   # PV: 1 score = mean 2 head
        lines = []
        for r, pm, ph, sc, gm, gh in zip(rows, mp, hp, score, my, hy):
            if pv:
                lines.append('%s\t%.4f' % (r['context_id'], float(sc)))
            else:
                lines.append('%s\t%.4f\t%.4f' % (r['context_id'], float(pm), float(ph)))
        fname = '%s-pred.tsv' % key          # vi du: en-nn-pred.tsv / de-pv-pred.tsv
        (sub / fname).write_text('\n'.join(lines), encoding='utf-8')
        msk = mask.astype(bool)
        if int(msk.sum()) > 0:
            if pv:
                rho = spearmanr(my[msk], score[msk]).correlation
                print('%s: %d rows | rho=%.4f' % (key, len(rows), rho))
            else:
                rho_m = spearmanr(my[msk], mp[msk]).correlation
                rho_h = spearmanr(hy[msk], hp[msk]).correlation
                print('%s: %d rows | rho mod=%.4f head=%.4f avg=%.4f'
                      % (key, len(rows), rho_m, rho_h, (rho_m + rho_h) / 2.0))
        else:
            print('%s: %d rows | khong co label (chi ghi prediction)' % (key, len(rows)))

    zip_p = work / 'submission.zip'          # archive dung format: cac *-pred.tsv o root
    with zipfile.ZipFile(zip_p, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(sub.glob('*-pred.tsv')):
            zf.write(f, arcname=f.name)
    print('submission files ->', sorted(f.name for f in sub.glob('*-pred.tsv')))
    print('zipped ->', zip_p)

In [ ]:
# --- TRIAL METRICS: compare saved predictions with labels when available ---
# This runs after the trial-prediction cell and is safe for unlabeled test files.
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

from src.config import Config
from src.data import load_trial

cfg_p = work / 'config.json'
cfg = Config() if not cfg_p.is_file() else Config(**json.loads(cfg_p.read_text(encoding='utf-8')))
trial_metrics = {}
for key, rows in load_trial(cfg).items():
    pred_path = work / 'submission' / f'{key}-pred.tsv'
    if not pred_path.is_file():
        print(f'{key}: prediction file missing: {pred_path.name}')
        continue

    gold = pd.DataFrame({
        'ContextID': [r['context_id'] for r in rows],
        'mod_avg': [r['mod_avg'] for r in rows],
        'head_avg': [r['head_avg'] for r in rows],
    })
    pred = pd.read_csv(pred_path, sep='\t', header=None, dtype=str)
    if bool(rows[0]['is_pv']):
        pred.columns = ['ContextID', 'pred']
        pred['pred'] = pd.to_numeric(pred['pred'], errors='coerce')
        scored = gold.merge(pred, on='ContextID', how='inner')
        mask = np.isfinite(scored.mod_avg) & np.isfinite(scored.pred)
        if mask.sum() > 1:
            y, p = scored.loc[mask, 'mod_avg'], scored.loc[mask, 'pred']
            trial_metrics[key] = {'n': int(mask.sum()), 'rho': float(spearmanr(y, p).statistic),
                                  'mse': float(mean_squared_error(y, p))}
    else:
        pred.columns = ['ContextID', 'pred_mod', 'pred_head']
        pred[['pred_mod', 'pred_head']] = pred[['pred_mod', 'pred_head']].apply(pd.to_numeric, errors='coerce')
        scored = gold.merge(pred, on='ContextID', how='inner')
        metric = {'n': int(len(scored))}
        for role in ('mod', 'head'):
            mask = np.isfinite(scored[f'{role}_avg']) & np.isfinite(scored[f'pred_{role}'])
            if mask.sum() > 1:
                y, p = scored.loc[mask, f'{role}_avg'], scored.loc[mask, f'pred_{role}']
                metric[f'{role}_rho'] = float(spearmanr(y, p).statistic)
                metric[f'{role}_mse'] = float(mean_squared_error(y, p))
        if len(metric) > 1:
            trial_metrics[key] = metric

metrics_path = work / 'submission' / 'trial_metrics.json'
metrics_path.write_text(json.dumps(trial_metrics, indent=2), encoding='utf-8')
print(json.dumps(trial_metrics, indent=2))
print('saved ->', metrics_path)

### LÆ°u Ã½
- TÃ­n hiá»‡u tá»‘t: `val_rho_mean` (rho trung bÃ¬nh Mod/Head trÃªn holdout 20%, split theo compound).
- Dedicated exits are always active: `gauss_ctx_mod=19`, `gauss_ctx_head=20`, and `gauss_ctx_pv=21,22`.
- Data mix: `TRAIN_MIX` = full (9825 rows, de-pv ~97% representation-only) / default (8335) / nn (6778).
- predict / train5 / resume chÆ°a wire trong `src/` (cháº¡y qua `mm/` cÅ© náº¿u cáº§n). CELL TRIAL chá»‰ tÃ¡i score holdout Ä‘á»ƒ xem prediction.